In [1]:
# Cell 1: setup
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))


In [2]:
# Cell: filter only by subject
from pathlib import Path
import pandas as pd
from src.embeddings.filter_rows import apply_filters

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

source_csv = ROOT / "data/processed/questions_clean_strict_rag_with_metadata.csv"
filtered_csv = ROOT / "data/intermediate/index_rows_subject_english.csv"

df = pd.read_csv(source_csv, dtype=str, keep_default_na=False)

filtered = apply_filters(
    df,
    language=None,
    question_type=None,
    multiple_choice=None,
    subjects=["ENGLISH"],   # only filter
    levels=[],
    quiz_title_contains=None,
    subjects_match_mode="any",
    levels_match_mode="any",
)

# optional safety: keep rows that have search_text
filtered = filtered[filtered["search_text"].fillna("").astype(str).str.strip() != ""].copy()

filtered.to_csv(filtered_csv, index=False)
print("Saved:", filtered_csv)
print("Rows:", len(filtered), "/", len(df))


Saved: /Users/macmini/Documents/work/softy/quiz-generator/data/intermediate/index_rows_subject_english.csv
Rows: 3626 / 8827


In [3]:
from src.embeddings.utils import load_runtime_config, dataclass_replace
from src.embeddings.build_vector_store import build_vector_store


config = load_runtime_config(ROOT / "configs/config.yaml")
config = dataclass_replace(
    config,
    clean_csv_path=filtered_csv,
    persist_directory=ROOT / "data/embeddings/chroma_db_subject_english_test",
    collection_name="quiz_questions_subject_english_test",
    reset_collection=True,
)

build_info = build_vector_store(config)
build_info


Profile language partition: profile='default' rows=3626/3626


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6908.00it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding 3626 rows from column 'search_text' with model 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'...
Indexed chunk rows 1-256/3626
Indexed chunk rows 257-512/3626
Indexed chunk rows 513-768/3626
Indexed chunk rows 769-1024/3626
Indexed chunk rows 1025-1280/3626
Indexed chunk rows 1281-1536/3626
Indexed chunk rows 1537-1792/3626
Indexed chunk rows 1793-2048/3626
Indexed chunk rows 2049-2304/3626
Indexed chunk rows 2305-2560/3626
Indexed chunk rows 2561-2816/3626
Indexed chunk rows 2817-3072/3626
Indexed chunk rows 3073-3328/3626
Indexed chunk rows 3329-3584/3626
Indexed chunk rows 3585-3626/3626
Indexed 3626 rows into collection 'quiz_questions_subject_english_test' (/Users/macmini/Documents/work/softy/quiz-generator/data/embeddings/chroma_db_subject_english_test).


/Users/macmini/Documents/work/softy/quiz-generator/src/embeddings/embedding_model.py:42: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  return self._model.get_sentence_embedding_dimension()


{'rows_indexed': 3626,
 'profile_name': 'default',
 'collection_name': 'quiz_questions_subject_english_test',
 'persist_directory': '/Users/macmini/Documents/work/softy/quiz-generator/data/embeddings/chroma_db_subject_english_test',
 'embedding_dimension': 384,
 'model_name': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
 'embedding_text_column': 'search_text',
 'distance_metric': 'cosine',
 'top_k': 5,
 'candidate_pool_size': 50,
 'subjects_match_mode': 'any',
 'levels_match_mode': 'any'}

In [4]:
from src.retrieval import Retriever

r = Retriever(config_path=config) 
results = r.retrieve(
    query="where whom pronouns",
    language="en",
    question_type="FILL_IN_THE_BLANKS",  # important if you want blanks
    top_k=6,
    max_distance=0.9,
)


print("count:", len(results))
for q in results:
    print("distance:", q.distance)
    print("search_text:", q.search_text)
    print("------")
    #pprint(q.metadata)      # full row metadata
    # or pprint(asdict(q))  # complete object


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9764.01it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6448.69it/s]
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


count: 1
distance: 0.7011086940765381
search_text: Quiz title: quiz Religion [TEST]
Question: He __ Ali
Choices: is ; s
Correct answer: is ; s
Subject: ENGLISH
Level: PRIMARY_SCHOOL_1ST_GRADE | PRIMARY_SCHOOL_3RD_GRADE
Question type: FILL_IN_THE_BLANKS
Language: en
------


/Users/macmini/Documents/work/softy/quiz-generator/src/retrieval/retriever.py:159: UserWarning: CSV/Chroma index drift detected: profile=default, expected_rows=3626, chroma_rows=6989. Rebuild vector store to realign.
  self._warn_if_index_drift()
/Users/macmini/Documents/work/softy/quiz-generator/src/retrieval/retriever.py:159: UserWarning: CSV/Chroma index drift detected: profile=arabic, expected_rows=0, chroma_rows=1838. Rebuild vector store to realign.
  self._warn_if_index_drift()
/var/folders/xg/ply3c5550g1f_bg2ytt7chrw0000gn/T/ipykernel_7891/565184156.py:4: UserWarning: Only 1/6 results returned after fallback. Consider relaxing filters or increasing top_k.
  results = r.retrieve(


In [16]:
# Cell: filter only by subject
from pathlib import Path
import pandas as pd
from src.embeddings.filter_rows import apply_filters

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

source_csv = ROOT / "data/processed/questions_clean_strict_rag_with_metadata.csv"
filtered_csv = ROOT / "data/intermediate/index_rows_subject_arabic.csv"

df = pd.read_csv(source_csv, dtype=str, keep_default_na=False)

filtered = apply_filters(
    df,
    language=None,
    question_type=None,
    multiple_choice=None,
    subjects=["ARABIC"],   # only filter
    levels=[],
    quiz_title_contains=None,
    subjects_match_mode="any",
    levels_match_mode="any",
)

# optional safety: keep rows that have search_text
filtered = filtered[filtered["search_text"].fillna("").astype(str).str.strip() != ""].copy()

filtered.to_csv(filtered_csv, index=False)
print("Saved:", filtered_csv)
print("Rows:", len(filtered), "/", len(df))


Saved: /Users/macmini/Documents/work/softy/quiz-generator/data/intermediate/index_rows_subject_arabic.csv
Rows: 1104 / 8827


In [17]:
from src.embeddings.utils import load_runtime_config, dataclass_replace
from src.embeddings.build_vector_store import build_vector_store


config = load_runtime_config(ROOT / "configs/config.yaml")
config = dataclass_replace(
    config,
    clean_csv_path=filtered_csv,
    persist_directory=ROOT / "data/embeddings/chroma_db_subject_arabic_test",
    collection_name="quiz_questions_subject_arabic_test",
    reset_collection=True,
)

build_info = build_vector_store(config)
build_info


ValueError: Missing required config key: vector_store.persist_directory

In [14]:
from src.retrieval import Retriever

r = Retriever(config_path=config) 
results = r.retrieve(
    query="أسئلة حول الفاعل في الجملة",
    language="ar",
    question_type="",  # important if you want blanks
    top_k=10,
    max_distance=0.9,
)


print("count:", len(results))
for q in results:
    print("distance:", q.distance)
    print("search_text:", q.search_text)
    print("------")
    #pprint(q.metadata)      # full row metadata
    # or pprint(asdict(q))  # complete object


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12817.95it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


count: 10
distance: 0.3735772371292114
search_text: Quiz title: الوُجُوبُ والإمكَانُ
Question: دَلَّ الفِعْلُ في الجُمْلَةِ التّالِيَةِ عَلَى: قَدْ بَلَغَ صاحِبُ الهِمَّةِ مَطلَبَهُ.
Choices: الوجوب ; الإمكان
Correct answer: الوجوب
Subject: ARABIC
Level: HIGH_SCHOOL_2ND_GRADE_COMPUTER_SCIENCE | HIGH_SCHOOL_2ND_GRADE_SCIENCE | HIGH_SCHOOL_2ND_GRADE_ECONOMICS_AND_SERVICE | HIGH_SCHOOL_2ND_GRADE_LETTRES
Question type: MULTIPLE_CHOICE
Language: ar
------
distance: 0.38310766220092773
search_text: Quiz title: المَجَازُ المُرْسَلُ
Question: مَا هِيَ العَلَاقَةُ بَيْنَ الأسَدِ وعَرِينِ الأسَدِ؟
Choices: المُسَبَّبِيَّةُ ; الجُزئِيَّةُ ; الحَالِّيَّةُ
Correct answer: الحَالِّيَّةُ
Subject: ARABIC
Level: HIGH_SCHOOL_1ST_GRADE
Question type: MULTIPLE_CHOICE
Language: ar
------
distance: 0.3890678882598877
search_text: Quiz title: الوُجُوبُ والإمكَانُ
Question: دَلَّ الفِعْلُ في الجُمْلَةِ التّالِيَةِ عَلَى: خَلِيلَيَّ بَلِّغَا رَقِيقَةَ القَلْبِ سَلَامِيَ.
Choices: الوُجوب ; الإمكان
Correct answ